# PPDBTAP Service Test Notebook

Automated tests for the ppdb TAP service.

## Imports

In [ ]:
from pyvo.dal.tap import TAPService
from astropy.table import Table
from lsst.rsp.utils import get_access_token
from lsst.rsp import RSPClient, get_service_url
import requests
from lsst.rsp import RSPDiscovery
from random import uniform

In [ ]:
discovery = RSPDiscovery("prompt")
service = discovery.get_tap_client()
assert service is not None
print(f"TAP service URL: {service.baseurl}")

## Asynchronous Query - Cone search

In [ ]:
radius = uniform(0.25, 0.30)
ra = 10
dec = -43
str_center_coords = str(ra) + ", " + str(dec)
str_radius = str(radius)

query = "SELECT * "\
        "FROM ppdb.DiaObject "\
         "WHERE CONTAINS(POINT('ICRS', ra, dec), "\
         "CIRCLE('ICRS', " + str_center_coords + ", " + str_radius + ")) = 1 "
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'], timeout=120)
if job.phase not in ('COMPLETED', 'ERROR'):
    raise TimeoutError(f"Job timed out after 2 minutes. Current phase: {job.phase}")
if job.phase == 'ERROR':
    job.raise_if_error()
results = job.fetch_result()
results.to_table()
